# import

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
from random import shuffle
from scipy.ndimage import gaussian_filter1d
from tqdm import tqdm
import multiprocessing
import os.path as osp
import os
import torch
from functools import partial
from multiprocessing import Pool

# Generate mixed sample

In [ ]:
import random
from itertools import combinations
from random import shuffle

def generate_mixed_sample_np(j,datadir,mode,threshold=0.1,
                          num_spectra=250,shift=0,noise_std=0.25):
    fns = os.listdir(datadir)
    fns.sort()
    n_class = len(fns)

    phases = used_combinations[mode][j]
    num_phase = len(phases)

    if mode=='train':
        select_range = range(shift+0,shift+int(num_spectra*0.8))
    elif mode=='val':
        select_range = range(shift+int(num_spectra*0.8),shift+int(num_spectra*0.9))
    else:
        select_range = range(shift+int(num_spectra*0.9),shift+num_spectra)

    portions = random.choice(frac_combinations_double) if num_phase == 2 else random.choice(frac_combinations_tripple)
    
    sample = np.zeros([4501,1])
    label = np.zeros([n_class,3])
    for i in range(num_phase):
        sample = sample + (aug_spec[phases[i]][random.choice(select_range),].reshape(4501,1))*portions[i]
        label[phases[i],i] = 1

    sample = sample/np.max(sample)*100
    sample = sample+np.random.normal(0, noise_std, (4501,1))

    # frac = np.zeros(3)
    # frac[:num_phase] = np.array(portions)

    return label

def bat_argu_process(datadir,savedir,combinations,mode,test_frac=0.1):

    datasize = len(combinations)
    num_cpu = multiprocessing.cpu_count()

    if not osp.exists(savedir):
        os.makedirs(savedir)

    def process(combinations,datasize):
        func = partial(generate_mixed_sample_np,datadir=datadir,mode=mode)
        print('generating')
        with Pool(num_cpu-2) as manager:
            dataset = list(tqdm(manager.imap(func,range(len(combinations))),total=datasize))

        print('transforming')   
        label = [i[0] for i in dataset]
        # frac = [i[1] for i in dataset]
        label = np.array(label).astype(np.float32)
        # frac = np.array(frac).astype(np.float32)
        label = torch.from_numpy(label)
        frac = torch.from_numpy(frac)
        return label,frac

    print('saving')
    label,frac = process(combinations,datasize)
    torch.save(label,osp.join(savedir,'%s_labels.pt'%mode))
    # torch.save(frac,osp.join(savedir,'%s_fractions.pt'%mode))

    return


num_phases = 136
dataname = 'zeolite'
# num_phases = 136
# dataname = 'limnti'
tripple_combinations = list(combinations(range(num_phases),3))
shuffle(tripple_combinations)
double_combinations = list(combinations(range(num_phases),2))
shuffle(double_combinations)

l_t = len(tripple_combinations)
l_d = len(double_combinations)
tripple_combinations = dict(train=tripple_combinations[:int(l_t*0.8)],val=tripple_combinations[int(l_t*0.8):int(l_t*0.9)],test=tripple_combinations[int(l_t*0.9):])
double_combinations = dict(train=double_combinations[:int(l_d*0.8)],val=double_combinations[int(l_d*0.8):int(l_d*0.9)],test=double_combinations[int(l_d*0.9):])

frac_combinations_double = [(0.1*a,1-0.1*a) for a in range(1,10)]
frac_combinations_tripple = [(0.1*a,0.1*b,1-0.1*a-0.1*b) for a in range(1,10) for b in range(1,10-a)]

used_combinations = dict()
for mode in ['train','val','test']:
    used_combinations[mode] = tripple_combinations[mode]+double_combinations[mode]


fns = os.listdir('../save/data/%s/aug_signal/'%dataname)
fns.sort()

aug_spec = []
for fn in fns:
    aug_spec.append(np.load('../save/data/%s/aug_signal/'%dataname+fn))

for mode in ['train','val','test']:
    bat_argu_process('../save/data/%s/aug_signal'%dataname,'../save/data/%s/mixed'%(dataname),used_combinations[mode],mode)
